In [ ]:
import os
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torch
from torchvision import transforms
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

class ImageDataset(Dataset):
    def __init__(self, csv_file, img_folder, transform=None):
        self.data = pd.read_csv(csv_file)
        self.img_folder = img_folder
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img_name = str(self.data.iloc[idx, 0])
        img_path_jpg = os.path.join(self.img_folder, img_name + '.jpg')
        img_path_png = os.path.join(self.img_folder, img_name + '.png')
        if os.path.exists(img_path_jpg):
            img_path = img_path_jpg
        elif os.path.exists(img_path_png):
            img_path = img_path_png
        else:
            raise FileNotFoundError(f"Image {img_name} not found with .jpg or .png extensions")
        image = Image.open(img_path).convert('RGB')
        label_jenis = self.data.iloc[idx, 1]
        label_warna = self.data.iloc[idx, 2]
        if self.transform:
            image = self.transform(image)
        labels = {'jenis': label_jenis, 'warna': label_warna}
        return image, labels

transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_csv_path = 'C:/Users/ijalaprilianto/Downloads/Penyisihan Hology Data Mining/train.csv'
train_img_folder = 'C:/Users/ijalaprilianto/Downloads/Penyisihan Hology Data Mining/train'
test_img_folder = 'C:/Users/ijalaprilianto/Downloads/Penyisihan Hology Data Mining/test'
output_csv_path = 'C:/Users/ijalaprilianto/Downloads/Penyisihan Hology Data Mining/submission.csv'

train_dataset = ImageDataset(
    csv_file=train_csv_path,
    img_folder=train_img_folder,
    transform=transform
)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

class CNNModel(nn.Module):
    def __init__(self):
        super(CNNModel, self).__init__()
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.fc1_jenis = nn.Linear(64 * 16 * 16, 128)
        self.fc2_jenis = nn.Linear(128, 2)
        self.fc1_warna = nn.Linear(64 * 16 * 16, 128)
        self.fc2_warna = nn.Linear(128, 5)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x)))
        x = x.view(-1, 64 * 16 * 16)
        jenis_out = F.relu(self.fc1_jenis(x))
        jenis_out = self.fc2_jenis(jenis_out)
        warna_out = F.relu(self.fc1_warna(x))
        warna_out = self.fc2_warna(warna_out)
        return jenis_out, warna_out

model = CNNModel()
criterion_jenis = nn.CrossEntropyLoss()
criterion_warna = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 10
for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        optimizer.zero_grad()
        jenis_pred, warna_pred = model(images)
        loss_jenis = criterion_jenis(jenis_pred, labels['jenis'])
        loss_warna = criterion_warna(warna_pred, labels['warna'])
        loss = loss_jenis + loss_warna
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    print(f'Epoch {epoch+1}, Loss: {running_loss/len(train_loader)}')

def predict(model, img_folder, transform, output_path):
    model.eval()
    test_images = [img for img in os.listdir(img_folder) if img.endswith(('.png', '.jpg', '.jpeg'))]
    predictions = []
    for img_name in test_images:
        img_path = os.path.join(img_folder, img_name)
        image = Image.open(img_path).convert('RGB')
        if transform:
            image = transform(image)
        image = image.unsqueeze(0)
        with torch.no_grad():
            jenis_pred, warna_pred = model(image)
            jenis_label = torch.argmax(jenis_pred, dim=1).item()
            warna_label = torch.argmax(warna_pred, dim=1).item()
            img_id = os.path.splitext(img_name)[0]
            predictions.append([int(img_id), jenis_label, warna_label])
    df = pd.DataFrame(predictions, columns=['id', 'jenis', 'warna'])
    df = df.sort_values(by='id')
    df.to_csv(output_path, index=False)
    print(f"Predictions saved to {output_path}")

predict(
    model=model,
    img_folder=test_img_folder,
    transform=transform,
    output_path=output_csv_path
)